In [3]:
import sys
import sklearn
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from functools import partial
import PIL
import PIL.Image

import torch
from torch import manual_seed as torch_manual_seed
import random
from torch.cuda import max_memory_allocated, set_device, manual_seed_all
from torch.backends import cudnn
from torch.nn import Module
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from typing import Any, Callable, List, Optional, Tuple, Type
from collections import namedtuple
from torch.utils.data import Dataset, DataLoader
from pytorchvideo.models.x3d import create_x3d
from torchvision.transforms import Compose, Lambda, Resize, Normalize
import torchvision.transforms._transforms_video as transforms

In [4]:
def setup_seed(seed):
    torch_manual_seed(seed)
    manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    cudnn.deterministic = True

SEED = 42
setup_seed(SEED)

In [ ]:
dataset_train = 
dataset_val = 
dataset_test = 

In [ ]:
loaders = {'train' : DataLoader(dataset_train, 
                         batch_size=32, 
                         shuffle=True, 
                         num_workers=1),
            'val'  : DataLoader(dataset_val, 
                         batch_size=32, 
                         shuffle=True, 
                         num_workers=1),
            'test'  : DataLoader(dataset_test, 
                         batch_size=32, 
                         shuffle=True, 
                         num_workers=1
}

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    train_loss, correct = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        if isinstance(pred, tuple) or isinstance(pred, GoogLeNetOutputs):
            pred = pred[0]
        loss = loss_fn(pred, y)
        train_loss += loss.item()
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    train_accuracies.append(correct/size * 100)
        

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            test_loss += loss.item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    
        test_loss /= num_batches
        correct /= size
        print(f"Validation Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    
    return correct, test_loss

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

x3d_model = create_x3d(
    input_clip_length=CLIP_LENGTH,
    input_crop_size=CROP_SIZE,
    model_num_class=NUM_CLASS,
    model_depth="M")
)
x3d_model = x3d_model.to(device)
optimizer = optim.SGD(x3d_model.parameters(), lr=0.0001, momentum=0.9)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train_losses = []
train_accuracies = []

for t in range(10):
    print(f"Epoch {t+1}\n-------------------------------")
    x3d_model.train()
    train_loop(loaders["train"], x3d_model, loss_fn, optimizer)
    x3d_model.eval()
    test_loop(loaders["val"], x3d_model, loss_fn)

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Training Loss", color='b')
plt.title('Training Loss per Batch')
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), train_accuracies, label="Training Accuracy", color='r')
plt.title('Training Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True)

In [ ]:
test_loop(loader_test, x3d_model, loss_fn)